# **ML_2. Supervised Learning**

### **1. Ответь на вопросы**

*i. Выведите аналитическое решение задачи регрессии. Используйте векторную форму уравнения.*   

*ii. Какие изменения произойдут в решении при добавлении L1- и L2-регуляризации к функции потерь?*  

*iii. Объясните, почему L1-регуляризация часто используется для отбора признаков. Почему после обучения модели остается много весов, равных 0?*    

*iV. Объясните, как можно использовать те же модели (линейная регрессия, гребневая регрессия и т. д.), но при этом обеспечить возможность учета нелинейных зависимостей*  

### **2. Введение**.  
*Выполните всю предварительную обработку данных из предыдущего урока.*

#### ***i. Импорт библиотек***

In [92]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

#### ***ii. Загрузка train и test датасетов. (read train and test parts)***

In [93]:
data_train = pd.read_json("../datasets/train.json")
print(f"""Количество строк: {data_train.shape[0]}
Количество столбцов: {data_train.shape[1]}""")

Количество строк: 49352
Количество столбцов: 15


In [94]:
data_test = pd.read_json("../datasets/test.json")
print(f"""Количество строк: {data_test.shape[0]}
Количество столбцов: {data_test.shape[1]}""")

Количество строк: 74659
Количество столбцов: 14


In [95]:
percentile_1 = data_train["price"].quantile(0.01)
percentile_99 = data_train["price"].quantile(0.99)

data_train = data_train[
    (data_train["price"] > percentile_1) & (data_train["price"] < percentile_99)
].copy()

price_lowest, price_highest = data_train["price"].min(), data_train["price"].max()


data_train["interest_level"] = data_train["interest_level"].map(
    {"low": 0, "medium": 1, "high": 2}
)
data_train.shape

(48343, 15)

In [96]:
data_test = data_test[
    (data_test["price"] >= price_lowest) & (data_test["price"] <= price_highest)
].copy()

bath_max = data_train["bathrooms"].max()
bed_max = data_train["bedrooms"].max()
data_test = data_test[
    (data_test["bathrooms"] <= bath_max) & (data_test["bedrooms"] <= bed_max)
].copy()

data_test.shape

(73175, 14)

### **3. Введение в анализ данных, часть 2**.  

#### ***i. Давайте создадим дополнительные характеристики для повышения качества модели. Рассмотрим столбец под названием «Features». Он содержит список основных особенностей текущей квартиры.***

In [97]:
cols = ["bathrooms", "bedrooms", "features"]
data_train[cols].head(10)

,bathrooms,bedrooms,features
4,1.0,1,"[Dining Room, Pre-War, Laundry in Building, Di..."
6,1.0,2,"[Doorman, Elevator, Laundry in Building, Dishw..."
9,1.0,2,"[Doorman, Elevator, Laundry in Building, Laund..."
10,1.5,3,[]
15,1.0,0,"[Doorman, Elevator, Fitness Center, Laundry in..."
16,1.0,3,"[Doorman, Elevator, Loft, Dishwasher, Hardwood..."
18,2.0,3,"[Fireplace, Laundry in Unit, Dishwasher, Hardw..."
19,1.0,0,"[Elevator, Laundry in Building, Dishwasher, Ha..."
23,0.0,1,[Hardwood Floors]
32,3.0,3,"[Cats Allowed, Dogs Allowed]"


#### ***ii. Удалите из столбца неиспользуемые символы ([,], ', ", и пробел).***

*после pd.read_json столбец features это не строка, а список Python. Поэтому для применения строковых методов нужно привести к строке.*

In [98]:
print(type(data_train["features"].iloc[0]))

<class 'list'>


In [99]:
data_train["features"].head(3)

4    [Dining Room, Pre-War, Laundry in Building, Di...
6    [Doorman, Elevator, Laundry in Building, Dishw...
9    [Doorman, Elevator, Laundry in Building, Laund...
Name: features, dtype: object

In [100]:
data_train["features"] = data_train["features"].astype(str)
data_train["features"].iloc[0]

"['Dining Room', 'Pre-War', 'Laundry in Building', 'Dishwasher', 'Hardwood Floors', 'Dogs Allowed', 'Cats Allowed']"

In [101]:
print(type(data_train["features"].iloc[0]))

<class 'str'>


In [102]:
for ch in ["[", "]", "'", '"', " "]:
    data_train["features"] = data_train["features"].str.replace(ch, "", regex=False)

data_train["features"].head(3)

4    DiningRoom,Pre-War,LaundryinBuilding,Dishwashe...
6    Doorman,Elevator,LaundryinBuilding,Dishwasher,...
9    Doorman,Elevator,LaundryinBuilding,LaundryinUn...
Name: features, dtype: str

In [103]:
print(
    "Всего пустых строк  в data_train датасете:", (data_train["features"] == "").sum()
)

Всего пустых строк  в data_train датасете: 3106


#### ***iii. Получите все значения в каждом списке и соберите результат в один большой список для всего набора данных. Для этого можно использовать DataFrame.iterrows().***

In [104]:
feature_list = []
for _, row in data_train.iterrows():
    if row["features"] != "":
        feature_list.extend(row["features"].split(","))

In [105]:
print("Длина списка:", len(feature_list))

Длина списка: 262377


In [106]:
feature_list[:5]

['DiningRoom', 'Pre-War', 'LaundryinBuilding', 'Dishwasher', 'HardwoodFloors']

#### ***iv. Сколько уникальных значений содержит список результатов?***

In [107]:
len(set(feature_list))

1529

#### ***v. Давайте познакомимся с новой библиотекой — Collections. С помощью этого пакета вы сможете эффективно получать количественную статистику по вашим данным.***

#### ***vi. Составьте список самых популярных функций из нашего обширного перечня и выберите 20 лучших на данный момент.***

In [108]:
from collections import Counter

In [109]:
feature_counts = Counter(feature_list)

In [110]:
top_20_pairs = feature_counts.most_common(20)
top_20_pairs

[('Elevator', 25375),
 ('HardwoodFloors', 23146),
 ('CatsAllowed', 23135),
 ('DogsAllowed', 21652),
 ('Doorman', 20479),
 ('Dishwasher', 20081),
 ('NoFee', 17793),
 ('LaundryinBuilding', 16082),
 ('FitnessCenter', 12989),
 ('Pre-War', 8971),
 ('LaundryinUnit', 8437),
 ('RoofDeck', 6417),
 ('OutdoorSpace', 5132),
 ('DiningRoom', 4890),
 ('HighSpeedInternet', 4223),
 ('Balcony', 2898),
 ('SwimmingPool', 2643),
 ('LaundryInBuilding', 2564),
 ('NewConstruction', 2504),
 ('Terrace', 2177)]

***Ранее на каггле был другой датасет. Сейчас обновленный. Больше, чем был до этого.***

*пустые строки не учитываю на моменте сбора feature_list

#### ***vii. Если все правильно, вы должны получить следующие значения: 'Elevator', 'CatsAllowed', 'HardwoodFloors', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace'.***

данные наименования фич совпадают с полученными (см.выше).

#### ***vi. Теперь создайте 20 новых признаков на основе 20 наиболее часто встречающихся значений: 1, если значение находится в столбце "Feature", в противном случае 0.***
*Now create 20 new features based on the top 20 values: 1 if the value is in the "Feature" column, otherwise 0.*

In [111]:
top_20 = [name for name, _ in top_20_pairs]

In [112]:
splits = data_train["features"].str.split(",")
type(splits)

pandas.Series

In [113]:
for feature in top_20:
    data_train[feature] = splits.apply(lambda lst: int(feature in lst))

In [114]:
data_train[top_20].head(3)

,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
4,0,1,1,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0
6,1,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
9,1,1,0,0,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0


#### ***ix. Расширьте наш набор признаков, добавив «ванные комнаты», «спальни», и создайте специальную переменную feature_list со всеми названиями признаков. Теперь у нас есть 22 значения. Все модели должны быть обучены на этих 22 признаках.***
Extend our feature set with 'bathrooms', 'bedrooms' and create a special variable feature_list with all feature names. Now we have 22 values. All models should be trained on these 22 features.

In [115]:
feature_list = top_20 + ["bathrooms", "bedrooms"]
print("Всего признаков:", len(feature_list))

Всего признаков: 22


### **4. Реализация моделей — Линейная регрессия**.  


#### ***i. Реализуйте класс Python для алгоритма линейной регрессии с двумя основными методами — fit и predict. Используйте стохастический градиентный спуск (SGD) для поиска оптимальных весов модели. Для лучшего понимания рекомендуем реализовать отдельные версии алгоритма с аналитическим решением и нестохастическим градиентным спуском.***
implement a Python class for a linear regression algorithm with two basic methods — fit and predict. Use stochastic gradient descent (SGD) to find optimal model weights. For better understanding, we recommend implementing separate versions of the algorithm with the analytical solution and non-stochastic gradient descent under the hood.

#### ***1) Алгоритм с аналитическим решением.***

Аналитическое решение задачи регрессии.  

$$
\boxed{
\hat{\boldsymbol{\beta}}
=
(X^TX)^{-1}X^T\mathbf y
}
$$

#### ***Как решается обратная матрица на практике или почему обычно применяют метод pinv (а не inv) или lstsq, и почему det не диагностика в float64***

В этом проекте X = 22 признака + столбец единиц = 23×23,
матрица полноранговая → XᵀX обратима, np.linalg.inv работает; np.linalg.pinv
здесь страховка (на полноранговой матрице она совпадает с inv). Незаменимой
pinv становится при вырожденной или плохо обусловленной матрице: в ML_1
полином 10 степени от двух базовых признаков (квадратные признаки там остались
только проверкой гипотезы, в модель не пошли) дал 66 столбцов ранга 44 < 66 →
XᵀX сингулярна, np.linalg.inv бросила бы LinAlgError: Singular matrix.

**Почему det не диагностика в float64** (математически det = 0 ⇔ вырожденность):
* масштаб: det(cA) = cⁿ·det(A), при n = 66 отличная матрица даёт det ~1e-46
  или 1e+46 просто из-за масштаба;
* overflow/underflow: det легко уходит в нуль или бесконечность;
* det не показывает степень обусловленности: у катастрофически плохой матрицы
  |det| = 1 (напр., diag(1e10, 1e-10)).

Корректная диагностика — cond = σ_max/σ_min: теряется ~log10(cond) знаков
точности, у float64 запас ~16 знаков, значит cond ~1e16 даёт чистый шум при
det = 1; cond(XᵀX) = cond(X)², поэтому на XᵀX смотрят особенно строго.

**Почему pinv.** Псевдообратная (Мура–Пенроуза) считается через SVD: нулевые
и близкие к нулю сингулярные значения отбрасываются → работает на вырожденных
и плохо обусловленных матрицах, возвращая псевдорешение (least squares)
минимальной нормы. sklearn.LinearRegression решает через SVD
(scipy.linalg.lstsq), поэтому обучилась бы даже на ранге 44.

**Практическое правило:** проверять cond, а не det; решать через pinv/lstsq,
а не через inv.

In [116]:
# ones — столбец (вектор) из единиц длиной len(data_train):
# по одной единице на каждый объект, чтобы свободный коэффициент (bias)
# стал обычным весом при столбце-единице
# ones = np.ones((len(data_train), 1))
# ones = np.ones((len(data_train), 1))
# ones

In [117]:
# np.hstack([data_train, ones])#

In [118]:
# минималистическая рабочая версия класса
class LinearRegressionAnalytic:
    def fit(self, X, y):
        X = np.asarray(
            X, dtype=float
        )  # np.array() cсоздает новую копию, а np.asarray копирует только когда это действительно нужно: список, DataFrame, другой dtype. Если вход уже float-массив, asarray вернет тот же объект без копирования.
        y = np.asarray(y, dtype=float)
        X_b = np.hstack([X, np.ones((len(X), 1))])  # столбец единиц для bias
        self.w_ = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        X_b = np.hstack([X, np.ones((len(X), 1))])
        return X_b @ self.w_

In [119]:
X_train = data_train[feature_list].to_numpy()
y_train = data_train["price"].to_numpy()

lr_analytic = LinearRegressionAnalytic().fit(X_train, y_train)
y_pred_analytic = lr_analytic.predict(X_train)

print("Размер вектора весов:", lr_analytic.w_.shape)
print("MAE train: ", round(mean_absolute_error(y_train, y_pred_analytic), 2))
print("RMSE train:", round(mean_squared_error(y_train, y_pred_analytic) ** 0.5, 2))

data_train["pred_lr_analytic"] = y_pred_analytic

Размер вектора весов: (23,)
MAE train:  708.74
RMSE train: 1027.26


In [120]:
import warnings


class LinearRegressionAnalytic:
    """класс для расчета линейной регрессии через аналитическое решение (нормальное уравнение) с опциями проверки ранга матрицы и ее обусловленности"""

    COND_THRESHOLD = 1e12  # для X^T X; эквивалентно cond(X) ~ 1e6

    def _prepare(self, X):
        """Приводит X к float-массиву и дописывает столбец единиц для bias."""
        X = np.asarray(X, dtype=float)
        return np.hstack([X, np.ones((len(X), 1))])

    def _solve(self, X_b, y):
        """Считает веса по готовой дизайн-матрице X_b и проверяет обусловленность."""
        XtX = X_b.T @ X_b
        self.cond_ = np.linalg.cond(XtX)
        self.rank_ = np.linalg.matrix_rank(XtX)

        if self.cond_ > self.COND_THRESHOLD:
            warnings.warn(
                f"X^T X плохо обусловлена: cond={self.cond_:.3e}, "
                f"rank={self.rank_} из {XtX.shape[0]}. "
                "Коэффициенты могут быть неустойчивыми, pinv вернул "
                "минимум-нормное решение.",
                UserWarning,
            )

        self.w_ = np.linalg.pinv(XtX) @ (X_b.T @ y)

    def fit(self, X, y):
        self._solve(self._prepare(X), np.asarray(y, dtype=float))
        return self

    def predict(self, X):
        return self._prepare(X) @ self.w_

    def fit_predict(self, X, y):
        X_b = self._prepare(X)
        self._solve(X_b, np.asarray(y, dtype=float))
        return X_b @ self.w_


In [ ]:
X_train = data_train[feature_list]
y_train = data_train["price"]

lr_analytic = LinearRegressionAnalytic().fit(X_train, y_train)

print("вектор весов:", lr_analytic.w_.shape)
print("cond(X^T X):", lr_analytic.cond_)
print("rank:", lr_analytic.rank_)

y_pred_train = lr_analytic.fit_predict(X_train, y_train)
print("MAE train:", round(mean_absolute_error(y_train, y_pred_train), 2))

вектор весов: (23,)
cond(X^T X): 483.4356716892379
rank: 23
MAE train: 708.74


* **rank = 23** равен числу столбцов, то есть X^T X полноранговая и обратимая:
  строгой мультиколлинеарности и вырожденности нет. Все столбцы линейно независимы, аналитическое решение существует и единственно.

* ***$cond(X^T X) ≈ 483.44$*** — матрица хорошо обусловлена: cond(X) = sqrt(483.44) ≈ 22,
  потеря точности при решении системы порядка $lg483 ≈ 2.7$  десятичных знака
  из ~16 доступных у float64. Это ничтожно мало, порог предупреждения (1e12)
  не превышен, поэтому решение численно устойчиво и ему можно доверять.

#### ***2) Алгоритм с нестохастическим градиентным спуском***

#### Математика нестохастического (полного) градиентного спуска

Главное отличие от аналитической версии: там веса получались за одну операцию
обращения матрицы, здесь это итеративный процесс.

**Задача.** Минимизируем функцию потерь по всей обучающей выборке:

$$ L(w) = \frac{1}{2n} \|Xw - y\|^2 = \frac{1}{2} \cdot \underbrace{\frac{1}{n}\|Xw - y\|^2}_{MSE} $$

Коэффициент $\frac{1}{2n}$ состоит из двух частей с разными ролями:
* $\frac{1}{n}$ — нормировка: сумма квадратов ошибок становится средней,
  поэтому потеря и градиент не зависят от размера выборки, градиент равен
  среднему градиентов по объектам, и lr переносим между датасетами;
* $\frac{1}{2}$ — удобство дифференцирования: производная квадрата дает
  множитель 2, который сокращается с этой половинкой.

Минимизация $L$ равносильна минимизации MSE: константный множитель не меняет
положение минимума. В аналитическом решении коэффициент вообще не требовался
(уравнение $\nabla L = 0$ не меняется от умножения на константу), в спуске он
масштабирует шаг, и этот масштаб поглощается lr.

**Градиент.** В аналитическом решении потеря уже была раскрыта и
продифференцирована по вектору весов (там он обозначен $\boldsymbol{\beta}$,
здесь $w$; это один и тот же вектор):

$$ \frac{\partial L}{\partial \boldsymbol{\beta}} = -2 X^T \mathbf{y} + 2 X^T X \boldsymbol{\beta}
\quad \text{для } L = \|\mathbf{y} - X\boldsymbol{\beta}\|^2 \text{ без нормирующего множителя} $$

Там производную приравняли к нулю и решили $X^T X \boldsymbol{\beta} = X^T \mathbf{y}$
относительно весов, так получилось точное решение за один шаг. Здесь производную
к нулю не приравниваем, а используем как направление шага. С множителем
$\frac{1}{2n}$ это та же производная, умноженная на константу:

$$ \nabla_w L(w) = \frac{1}{2n}\left(2 X^T X w - 2 X^T \mathbf{y}\right) = \frac{1}{n} X^T (X w - \mathbf{y}) $$

Градиент указывает направление наибыстрейшего роста функции потерь, поэтому
шаг делаем в противоположную сторону. Именно этот градиент считает код класса:
grad = X_b.T @ residual / n.

**Обновление весов.** На каждой итерации:

$$ w \leftarrow w - lr \cdot \nabla_w L(w) = w - \frac{lr}{n} X^T (X w - \mathbf{y}) $$

Метод называется нестохастическим (full-batch), потому что градиент вычисляется
точно по всем n объектам выборки на каждом шаге, а не по одному случайному
объекту (SGD) и не по случайному подмножеству (mini-batch).

**Критерии остановки.** Функция потерь выпукла (квадратичная форма с
положительной полупределенной матрицей $X^T X / n$), поэтому ее минимум
характеризуется условием $\nabla L(w) = 0$. Значит, итерации прекращаем, когда
норма градиента опускается ниже порога tol (условие оптимальности первого
порядка), либо когда исчерпан лимит итераций max_iter — это страховка от
бесконечного цикла, если сходимость медленная (маленький lr или плохо
обусловленная матрица).

**Скорость сходимости.** На квадратичной функции спуск сходится линейно,
и число итераций зависит от числа обусловленности $cond(X^T X)$: чем оно больше,
тем медленнее спуск и тем сильнее «зигзаги» в узком овраге. Именно поэтому
класс внутри стандартизует признаки и цель: после стандартизации овраг
выравнивается, и один и тот же lr работает для всех координат.

<details>
<summary>Математика нестохастического (полного) градиентного спуска</summary>

Главное отличие от аналитической версии: там веса получались за одну операцию
обращения матрицы, здесь это итеративный процесс.

**Задача.** Минимизируем функцию потерь по всей обучающей выборке:

$$ L(w) = \frac{1}{2n} \|Xw - y\|^2 = \frac{1}{2} \cdot \underbrace{\frac{1}{n}\|Xw - y\|^2}_{MSE} $$

Коэффициент $\frac{1}{2n}$ состоит из двух частей с разными ролями:
* $\frac{1}{n}$ — нормировка: сумма квадратов ошибок становится средней,
  поэтому потеря и градиент не зависят от размера выборки, градиент равен
  среднему градиентов по объектам, и lr переносим между датасетами;
* $\frac{1}{2}$ — удобство дифференцирования: производная квадрата дает
  множитель 2, который сокращается с этой половинкой.

Минимизация $L$ равносильна минимизации MSE: константный множитель не меняет
положение минимума. В аналитическом решении коэффициент вообще не требовался
(уравнение $\nabla L = 0$ не меняется от умножения на константу), в спуске он
масштабирует шаг, и этот масштаб поглощается lr.

**Градиент.** В аналитическом решении потеря уже была раскрыта и
продифференцирована по вектору весов (там он обозначен $\boldsymbol{\beta}$,
здесь $w$; это один и тот же вектор):

$$ \frac{\partial L}{\partial \boldsymbol{\beta}} = -2 X^T \mathbf{y} + 2 X^T X \boldsymbol{\beta}
\quad \text{для } L = \|\mathbf{y} - X\boldsymbol{\beta}\|^2 \text{ без нормирующего множителя} $$

Там производную приравняли к нулю и решили $X^T X \boldsymbol{\beta} = X^T \mathbf{y}$
относительно весов, так получилось точное решение за один шаг. Здесь производную
к нулю не приравниваем, а используем как направление шага. С множителем
$\frac{1}{2n}$ это та же производная, умноженная на константу:

$$ \nabla_w L(w) = \frac{1}{2n}\left(2 X^T X w - 2 X^T \mathbf{y}\right) = \frac{1}{n} X^T (X w - \mathbf{y}) $$

Градиент указывает направление наибыстрейшего роста функции потерь, поэтому
шаг делаем в противоположную сторону. Именно этот градиент считает код класса:
grad = X_b.T @ residual / n.

**Обновление весов.** На каждой итерации:

$$ w \leftarrow w - lr \cdot \nabla_w L(w) = w - \frac{lr}{n} X^T (X w - \mathbf{y}) $$

Метод называется нестохастическим (full-batch), потому что градиент вычисляется
точно по всем n объектам выборки на каждом шаге, а не по одному случайному
объекту (SGD) и не по случайному подмножеству (mini-batch).

**Критерии остановки.** Функция потерь выпукла (квадратичная форма с
положительной полупределенной матрицей $X^T X / n$), поэтому ее минимум
характеризуется условием $\nabla L(w) = 0$. Значит, итерации прекращаем, когда
норма градиента опускается ниже порога tol (условие оптимальности первого
порядка), либо когда исчерпан лимит итераций max_iter — это страховка от
бесконечного цикла, если сходимость медленная (маленький lr или плохо
обусловленная матрица).

**Скорость сходимости.** На квадратичной функции спуск сходится линейно,
и число итераций зависит от числа обусловленности $cond(X^T X)$: чем оно больше,
тем медленнее спуск и тем сильнее «зигзаги» в узком овраге. Именно поэтому
класс внутри стандартизует признаки и цель: после стандартизации овраг
выравнивается, и один и тот же lr работает для всех координат.

</details>


In [ ]:
class LinearRegressionGD:
    """Линейная регрессия через полный (по всей выборке, нестохастический) градиентный спуск.
        L(w) = 1/(2n) * ||X w - y||^2,  grad = X^T (X w - y) / n,
        w <- w - lr * grad.
    X и y стандартизуются, что дает устойчивый шаг и интерпретируемые tol и loss.
    """

    def __init__(self, lr=0.1, tol=1e-8, max_iter=10000):
        self.lr = lr
        self.tol = tol
        self.max_iter = max_iter

    def _prepare(self, X, fit=False):
        X = np.asarray(X, dtype=float)
        if fit:
            self.mean_ = X.mean(axis=0)
            self.std_ = X.std(axis=0)
            self.std_[self.std_ == 0] = (
                1.0  # у константного столбца std=0: меняем на 1, чтобы не делить на ноль (столбец станет нулевым, без nan)
            )
        X_s = (X - self.mean_) / self.std_
        return np.hstack([X_s, np.ones((len(X_s), 1))])

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.y_mean_ = y.mean()
        self.y_std_ = y.std() or 1.0
        y_s = (y - self.y_mean_) / self.y_std_

        X_b = self._prepare(X, fit=True)
        n = len(y_s)
        self.w_ = np.zeros(X_b.shape[1])
        self.loss_history_ = []

        for it in range(self.max_iter):
            residual = X_b @ self.w_ - y_s  # (X w - y)
            grad = X_b.T @ residual / n  # grad = X^T (X w - y) / n
            self.w_ -= self.lr * grad  # w = w - lr * grad
            self.loss_history_.append(np.mean(residual**2) / 2)
            if np.linalg.norm(grad) < self.tol:
                break

        self.n_iter_ = it + 1
        return self

    def predict(self, X):
        y_s = self._prepare(X) @ self.w_
        return y_s * self.y_std_ + self.y_mean_

-------------------------

### Что сохраняет self.loss_history_.append(np.mean(residual ** 2) / 2)

В начале итерации посчитан вектор остатков $r = X_b w - y_s$ длины $n$; тот же вектор используется в градиенте $\nabla L = X_b^T r / n$. Строка вычисляет значение потери на текущих весах:

$$ L(w) = \frac{1}{2n}\|r\|_2^2 = \frac{1}{2} \cdot \underbrace{\frac{1}{n}\sum_{i=1}^{n} r_i^2}_{MSE} $$

* `residual ** 2` — поэлементные квадраты $(r_1^2, \dots, r_n^2)$;
* `np.mean(...)` — среднее по объектам, $MSE = \frac{1}{n}\sum r_i^2$;
* `/ 2` — половина MSE, чтобы хранить ту же $L = \frac{1}{2n}\|r\|_2^2$, которую дифференцируем;
* `.append(...)` — значение в историю сходимости, по одному числу на итерацию.

Почему именно эта $L$: шаг $w \leftarrow w - lr \cdot \nabla L$ использует ее градиент, значит, плато кривой соответствует минимуму именно ее. Константный множитель не сдвигает минимум, поэтому /2 влияет только на масштаб кривой, не на оптимизацию.

Чтение истории: монотонное убывание — норма; плато — сходимость; рост или nan — расходимость, lr слишком большой.

Нюансы: остаток берется до обновления весов, поэтому `loss_history_[it]` это потеря на весах начала итерации (первый элемент — при $w = 0$). Значения в стандартизованных единицах; пересчет в исходные: $MSE_{orig} = 2 \cdot L \cdot y\_std\_{}^2$.

Числовой пример: остатки $(-1, -1, -1)$, $n = 3$: квадраты $(1, 1, 1)$, среднее $1$, половина $0.5$; сверка $L = \frac{1}{2 \cdot 3} \cdot 3 = 0.5$.

### Что делает строка if np.linalg.norm(grad) < self.tol: break

Градиент на текущих весах: $ \nabla L(w) = \frac{1}{n} X_b^T r $, где $ r = X_b w - y_s $.
Его евклидова норма: $ \|\nabla L(w)\|_2 = \sqrt{g_0^2 + g_1^2 + \dots + g_d^2} $ - одно число, суммарная крутизна потерь в точке.

Условие останова: $ \|\nabla L(w)\|_2 < tol $. Для выпуклой гладкой функции минимум характеризуется условием оптимальности первого порядка $ \nabla L(w^*) = 0 $; точный ноль в float64 недостижим, поэтому останавливаемся при пренебрежимо малой норме. Следующий шаг изменил бы веса на $ \|\Delta w\| = lr \cdot \|\nabla L(w)\| < lr \cdot tol $, то есть вне значимых знаков прогноза.

Команда break досрочно выходит из цикла итераций; в self.n_iter_ остается номер итерации останова. Если условие не сработало ни разу, цикл докручивает до max_iter, и n_iter_ == max_iter служит сигналом несходимости (мал lr или лимит).

Норма агрегирует компоненты: из $ \|g\|_2 < tol $ следует $ |g_i| < tol $ для всех i, поэтому одной проверки достаточно.

**Числа.** $ X = \begin{pmatrix} 1 & 1 \\ 1 & 2 \\ 1 & 3 \end{pmatrix} $, $ y = (2,3,4) $, $ r = Xw - y $.
Точка $ w = (2,1) $: $ r = (1,1,1) $, $ g = X^T r / 3 = (1, 2) $, $ \|g\|_2 = \sqrt{5} \approx 2.236 \gg 10^{-8} $ — не останавливаемся, шагаем.
Точка $ w = (1+10^{-9}, 1) $: $ r = (10^{-9}, 10^{-9}, 10^{-9}) $, $ g = (10^{-9}, 2\cdot10^{-9}) $, $ \|g\|_2 \approx 2.24\cdot10^{-9} < 10^{-8} $ - break, сходимость достигнута.

-----------------------

In [126]:
gd = LinearRegressionGD(lr=0.1, tol=1e-8, max_iter=10000).fit(X_train, y_train)

print("итераций:", gd.n_iter_)
print("MAE train:", round(mean_absolute_error(y_train, gd.predict(X_train)), 2))
print(
    "max |pred_gd - pred_analytic|:",
    np.abs(gd.predict(X_train) - lr_analytic.predict(X_train)).max(),
)

итераций: 2024
MAE train: 708.74
max |pred_gd - pred_analytic|: 0.00038172230779309757


#### ***3) Стохастический градиентный спуск (SGD)***

#### ***ii. Что такое детерминированная модель? Сделайте SGD детерминированным.***
What is determenistic model? Make SGD determenistic.

#### ***iii. Определите коэффициент R² и реализуйте функцию для его вычисления.***
Define the R squared (R2) coefficient and implement a function to calculate it.

#### ***iv. С помощью вашего алгоритма сделайте прогнозы и оцените модель, используя метрики MAE, RMSE и R2.***
Make predictions with your algorithm and estimate the model with MAE, RMSE and R2 metrics.

#### ***v. Инициализируйте функцию LinearRegression() из sklearn.linear_model, обучите модель и выполните прогнозирование для обучающей и тестовой частей, как в предыдущем уроке.***
Initialize LinearRegression() from sklearn.linear_model, fit the model, and predict the training and test parts as in the previous lesson.

#### ***vi. Сравните показатели качества и убедитесь, что разница невелика (между вашей реализацией и sklearn).***
Compare the quality metrics and make sure the difference is small (between your implementations and sklearn).

#### ***vii. Сохраните метрики, как и в предыдущем уроке, в таблице со столбцами model, train, test для таблицы MAE, таблицы RMSE и коэффициента R2.***
Store the metrics as in the previous lesson in a table with columns model, train, test for MAE table, RMSE table, and R2 coefficient.